# BÀI THỰC HÀNH LAB 03: NGỮ PHÁP ĐỒ HỌA & THAO TÁC DỮ LIỆU THEO NHÓM
## Môn học: Xác suất Thống kê (UET.MAT1052)
### Môi trường: Python 3, Pandas & plotnine

---
**Họ và tên sinh viên:** .....................................................  
**Mã số sinh viên (MSV):** .....................................................  
**Lớp học phần:** .....................................................  

> **Hướng dẫn:** Chạy các ô code bằng tổ hợp phím `Shift + Enter`. Hoàn thành các ô có chú thích `# TODO:` và phân tích nhận xét kết quả thu được.

### 1. Khởi động môi trường và thư viện

In [ ]:
import numpy as np
import pandas as pd
from plotnine import (
    ggplot,
    aes,
    geom_point,
    geom_line,
    geom_col,
    geom_hline,
    labs,
    theme_minimal,
    scale_color_manual,
    scale_fill_manual,
)

### 2. Khám phá dữ liệu Số ca sinh nở tại Hoa Kỳ (`us_births.csv`)
1. Đọc dữ liệu từ file `du-lieu/us_births.csv`.
2. Tính tổng số ca sinh theo từng năm và so sánh giữa biểu đồ Điểm (`geom_point`) và Đường (`geom_line`).

In [ ]:
df_births = pd.read_csv("du-lieu/us_births.csv")
print("Kích thước bảng:", df_births.shape)
display(df_births.head())

# Tổng số ca sinh theo năm
yearly_births = df_births.groupby("year")["births"].sum().reset_index()

# 1. Dạng Điểm: Tôn trọng từng giá trị riêng rẽ
p_point = (
    ggplot(yearly_births, aes(x="year", y="births"))
    + geom_point(color="#0284c7", size=3)
    + labs(title="Dạng Điểm: Tôn trọng từng giá trị riêng rẽ", x="Năm", y="Tổng số ca sinh")
    + theme_minimal()
)
print(p_point)

# 2. Dạng Đường: Nhấn mạnh tính liên tục & xu hướng
p_line = (
    ggplot(yearly_births, aes(x="year", y="births"))
    + geom_line(color="#ea580c", size=1.2)
    + geom_point(color="#ea580c", size=2)
    + labs(title="Dạng Đường: Nhấn mạnh tính liên tục & xu hướng", x="Năm", y="Tổng số ca sinh")
    + theme_minimal()
)
print(p_line)

### 3. Giải mã hai dải dữ liệu song song (Năm 1988)
Thêm ánh xạ màu sắc (`color="nhom_ngay"`) theo biến phân loại ngày trong tuần vs ngày cuối tuần.

In [ ]:
births_1988 = df_births.query("year == 1988").copy()
births_1988["nhom_ngay"] = births_1988["day_of_week"].apply(
    lambda x: "Cuối tuần (T7, CN)" if x in [6, 7] else "Ngày thường (T2-T6)"
)

p_bands = (
    ggplot(births_1988, aes(x="day_of_year", y="births", color="nhom_ngay"))
    + geom_point(size=2, alpha=0.8)
    + scale_color_manual(values={"Ngày thường (T2-T6)": "#0284c7", "Cuối tuần (T7, CN)": "#f43f5e"})
    + labs(
        title="Giải Mã Hai Dải Dữ Liệu: Tác Động Của Ngày Làm Việc Đến Số Ca Sinh (1988)",
        x="Ngày trong năm (1 - 366)",
        y="Số ca sinh mỗi ngày",
        color="Phân loại ngày",
    )
    + theme_minimal()
)
print(p_bands)

### 4. Ngữ pháp đồ họa đa biến trên chim cánh cụt (`penguins.csv`)
Phân biệt giữa Mapping (`color="species"`, `shape="species"` trong `aes`) và Setting (`size=3`, `alpha=0.85` trong `geom_point`).

In [ ]:
df_penguins = pd.read_csv("du-lieu/penguins.csv").dropna(subset=["bill_length_mm", "bill_depth_mm", "species"])

p_penguins = (
    ggplot(df_penguins, aes(x="bill_length_mm", y="bill_depth_mm", color="species", shape="species"))
    + geom_point(size=3, alpha=0.85)
    + labs(
        title="Tương Quan Chiều Dài và Độ Sâu Mỏ Giữa Các Loài Chim Cánh Cụt",
        x="Chiều dài mỏ (mm)",
        y="Độ sâu mỏ (mm)",
        color="Loài chim",
        shape="Loài chim",
    )
    + theme_minimal()
)
print(p_penguins)

### 5. Dây chuyền `groupby` và kiểm tra chéo
Tính các đặc trưng trung bình và độ lệch chuẩn của các loài chim cánh cụt.

In [ ]:
penguins_stats = (
    df_penguins
    .groupby("species", observed=True)
    .agg(
        so_luong=("species", "count"),
        tb_chieu_dai=("bill_length_mm", "mean"),
        do_lech_dai=("bill_length_mm", "std"),
        tb_do_sau=("bill_depth_mm", "mean"),
        do_lech_sau=("bill_depth_mm", "std")
    )
    .reset_index()
)
display(penguins_stats.round(2))

### 6. Thực nghiệm kiểm chứng Nghịch lý Simpson
So sánh tỷ lệ cá thể có khối lượng vượt 4000g: Tóm tắt gộp ($50.3\%$) vs Tóm tắt có điều kiện theo loài.

In [ ]:
df_valid_mass = df_penguins.dropna(subset=["body_mass_g"]).copy()
df_valid_mass["tren_4000g"] = df_valid_mass["body_mass_g"] > 4000

ty_le_gop = df_valid_mass["tren_4000g"].mean()
print(f"Tỷ lệ gộp toàn bộ cá thể > 4000g: {ty_le_gop * 100:.1f}%")

ty_le_theo_loai = (
    df_valid_mass
    .groupby("species")["tren_4000g"]
    .agg(
        so_ca_tren_4000="sum",
        tong_so="count",
        ty_le="mean"
    )
    .reset_index()
)
ty_le_theo_loai["ty_le_phan_tram"] = (ty_le_theo_loai["ty_le"] * 100).round(1)
display(ty_le_theo_loai)

# Trực quan hóa so sánh bằng plotnine
p_simpson = (
    ggplot(ty_le_theo_loai, aes(x="species", y="ty_le_phan_tram", fill="species"))
    + geom_col(width=0.6)
    + geom_hline(yintercept=ty_le_gop * 100, color="red", linetype="dashed", size=1)
    + scale_fill_manual(values={"Adelie": "#94a3b8", "Chinstrap": "#94a3b8", "Gentoo": "#10b981"})
    + labs(
        title="Nghịch Lý Tóm Tắt Gộp: Loài Gentoo Phân Hóa Tuyệt Đối",
        subtitle=f"Đường đỏ nét đứt: Tỷ lệ gộp chung ({ty_le_gop*100:.1f}%)",
        x="Loài chim cánh cụt",
        y="Tỷ lệ cá thể nặng > 4000g (%)",
        fill="Loài",
    )
    + theme_minimal()
)
print(p_simpson)